# DuckDB + Snowflake Horizon Iceberg REST Catalog (HIRC)

Query Snowflake-managed **Dynamic Iceberg Tables** directly from DuckDB — no data copy,
no ETL. DuckDB talks to Snowflake's Polaris-based REST catalog endpoint and reads
Iceberg data files from S3 using vended credentials.

**This notebook runs in Google Colab, Jupyter, or any Python environment.**
The only dependency is `duckdb`.

## Prerequisites

Create the SA_USER, ROLE and PAT required to connect DuckDB with Snowflake via HIRC.

Open and import the SQL https://github.com/Snowflake-Labs/sfguide-lakehouse-iceberg-production-pipelines/blob/main/snowflake/lab/05_duckdb_hirc_setup.sql into your Snowflake workspace.

### Create SA User, Rol and PAT

Uncomment and run the Stored Procedure `setup_duckdb_service_account`. Be sure to capture the PAT from the output.

### What you need to do before running this notebook:

1. **Generate a Programmatic Access Token (PAT)** in Snowsight:
   - Click your user menu (bottom-left) → **Programmatic Access Tokens** → **Generate**
   - **Scope the token to the role** your instructor provided (e.g. `duckdb_silver_reader`)
   - Copy the token — you'll paste it below

2. **Gather these 4 values** (your instructor will provide the first three):

   | Value | Example |
   |-------|---------|
   | Snowflake Account URL | `https://myorg-myaccount.snowflakecomputing.com` |
   | Role name | `duckdb_silver_reader` |
   | Database name | `balloon_silver` |
   | PAT token | *(you just generated this)* |

> **Note:** Do NOT apply Horizon grants yet — we'll demonstrate what happens
> without them first, then fix it.

In [ ]:
!pip install --force-reinstall --no-cache-dir duckdb==1.5.2

In [ ]:
import duckdb

print(duckdb.__version__)

## Enter your credentials

Run the cell below and enter the values when prompted.
The PAT is collected via `getpass` so it won't be displayed.

In [ ]:
import re
from google.colab import userdata

SNOWFLAKE_ACCOUNT_URL = (
    input("Snowflake account URL (e.g. https://org-account.snowflakecomputing.com): ")
    .strip()
    .rstrip("/")
)
PAT_TOKEN = userdata.get("DUCKDB_SA_PAT")
SA_ROLE = (
    input("Role name [duckdb_silver_reader]: ").strip().upper()
    or "duckdb_silver_reader".upper()
)
DATABASE = (
    input("Database name [balloon_silver]: ").strip().upper()
    or "balloon_silver".upper()
)

if not SNOWFLAKE_ACCOUNT_URL.startswith("http"):
    SNOWFLAKE_ACCOUNT_URL = "https://" + SNOWFLAKE_ACCOUNT_URL


# Snowflake account URLs use hyphens between org and account locator, but the
# account identifier portion may contain underscores that Snowsight displays as
# hyphens. Normalize: replace hyphens with underscores AFTER the org-account separator.
# e.g. https://myorg-my_account_locator.snowflakecomputing.com
# The org-name separator is always the FIRST hyphen in the hostname.
def normalize_account_url(url: str) -> str:
    """Keep org-account hyphen, convert remaining hyphens in host to underscores."""
    match = re.match(r"(https?://)([^/]+)(.*)", url)
    if not match:
        return url
    scheme, host, rest = match.groups()
    # Split on first hyphen (org separator) then fix the remainder
    parts = host.split("-", 1)
    if len(parts) == 2:
        host = parts[0] + "-" + parts[1].replace("-", "_")
    return scheme + host + rest


SNOWFLAKE_ACCOUNT_URL = normalize_account_url(SNOWFLAKE_ACCOUNT_URL)
CATALOG_URI = SNOWFLAKE_ACCOUNT_URL.lower() + "/polaris/api/catalog"
CATALOG_NAME = (
    DATABASE.upper()
)  # HIRC is case-sensitive; Snowflake stores DB names uppercase

print(f"\nCatalog URI : {CATALOG_URI}")
print(f"Database    : {DATABASE}")
print(f"Catalog     : {CATALOG_NAME}")
print(f"Role        : {SA_ROLE}")
print(f"PAT length: {len(PAT_TOKEN)}, starts with: {PAT_TOKEN[:10]}...")

## Verify HIRC API

In [ ]:
import subprocess
from google.colab import userdata

PAT = userdata.get("DUCKDB_SA_PAT")
result = subprocess.run(
    [
        "curl",
        "-s",
        "-w",
        "\nHTTP_STATUS: %{http_code}",
        "-X",
        "POST",
        f"{CATALOG_URI}/v1/oauth/tokens",
        "--header",
        "Content-Type: application/x-www-form-urlencoded",
        "--data-urlencode",
        "grant_type=client_credentials",
        "--data-urlencode",
        f"scope=session:role:{SA_ROLE}",
        "--data-urlencode",
        f"client_secret={PAT}",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)

## Load DuckDB Extensions

DuckDB's `iceberg` extension provides the REST catalog client.
`httpfs` enables reading remote Parquet/Iceberg data files via S3.

In [ ]:
import duckdb

conn = duckdb.connect()
conn.execute("INSTALL iceberg;")
conn.execute("LOAD iceberg;")
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")
print("Extensions loaded.")

## Connect to Horizon Iceberg REST Catalog

DuckDB authenticates via the PAT using OAuth2 client credentials flow,
then attaches the Snowflake database as a local DuckDB catalog.

> **Important (DuckDB 1.2+):**
> - The **ENDPOINT** must be set in the **SECRET** (not in ATTACH).
> - **ATTACH** uses the **catalog/database name** (e.g. `BALLOON_SILVER`), not the URL.
> - The catalog name must be **UPPERCASE** — HIRC/Polaris is case-sensitive.

In [ ]:
import traceback

secret_sql = f"""
  CREATE OR REPLACE SECRET iceberg_pat_secret (
    TYPE iceberg,
    CLIENT_ID '',
    CLIENT_SECRET '{PAT_TOKEN}',
    ENDPOINT '{CATALOG_URI}',
    OAUTH2_SERVER_URI '{CATALOG_URI}/v1/oauth/tokens',
    OAUTH2_GRANT_TYPE 'client_credentials',
    OAUTH2_SCOPE 'session:role:{SA_ROLE}'
  );
"""
attach_sql = f"""
  ATTACH '{CATALOG_NAME}' AS {DATABASE} (
    TYPE iceberg,
    SECRET iceberg_pat_secret
  );
"""

# Detach if previously attached (safe to fail on first run)
try:
    conn.execute("USE memory")
    conn.execute(f"DETACH {DATABASE}")
except Exception:
    pass

# Now attach fresh
try:
    conn.execute(secret_sql)
    conn.execute(attach_sql)
    print(f"Attached {CATALOG_NAME} (alias: {DATABASE}) via HIRC.")
except Exception:
    traceback.print_exc()

## Discover Tables

In [ ]:
try:
    conn.execute(f"USE {DATABASE}.SILVER")
    rows = conn.execute("SHOW TABLES").fetchall()
    print(f"Found {len(rows)} table(s) in {DATABASE}.SILVER:")
    for r in rows:
        print(f"  {r[0]}")
except Exception:
    traceback.print_exc()

## First Query Attempt (expected to FAIL)

Let's try querying a silver Dynamic Iceberg Table.
**This will fail** because the Horizon grants have not been applied yet.

In [ ]:
try:
    df = conn.execute(f"""
        SELECT player, total_score, bonus_pops, last_event_ts
        FROM {CATALOG_NAME}.SILVER.DT_PLAYER_LEADERBOARD
        ORDER BY total_score DESC NULLS LAST
        LIMIT 5
    """).df()
    print("Query succeeded (grants were already in place):")
    print(df.to_string(index=False))
except Exception as e:
    print(f"EXPECTED FAILURE: {e}")
    print("\n↓ See the next cell for why this failed and how to fix it.")

## Why did that fail?

HIRC requires **explicit grants** on the role. Specifically:

- `GRANT SELECT ON ALL TABLES` **silently skips Dynamic Tables** in Snowflake
- You must use `ON ALL DYNAMIC TABLES` and `ON FUTURE DYNAMIC TABLES`

### Run these grants in Snowsight (as ACCOUNTADMIN)

Replace `SUMMIT26_AR103_BALLOON_SILVER` and `SUMMIT26_AR103_DUCKDB_SILVER_READER` with your values:

```sql
CALL grant_duckdb_read_access();
```

**Verify in Snowsight:**
```sql
SHOW GRANTS TO ROLE SUMMIT26_AR103_DUCKDB_SILVER_READER;
```
You should see `SELECT` on each of the five `DT_*` dynamic tables plus `USAGE` on the database and schema.

---

**Once the grants are applied, run the next cell to retry.**

## Retry After Grants

Now that Horizon grants are in place, the same query should succeed.
We detach and re-attach so DuckDB picks up the updated permissions.

In [ ]:
# Re-attach to pick up the new grants
try:
    conn.execute(f"DETACH {DATABASE}")
except Exception:
    pass  # ignore if not attached

conn.execute(secret_sql)
try:
    conn.execute(attach_sql)
except Exception:
    pass  # ignore if already attached

print(f"Re-attached {CATALOG_NAME} via HIRC.\n")

df = conn.execute(f"""
    SELECT player, total_score, bonus_pops, last_event_ts
    FROM {CATALOG_NAME}.SILVER.DT_PLAYER_LEADERBOARD
    ORDER BY total_score DESC NULLS LAST
    LIMIT 10
""").df()
print("dt_player_leaderboard:")
print(df.to_string(index=False))

## Query All Silver Dynamic Iceberg Tables

Now let's query all five silver DTs to confirm full access.

In [ ]:
# dt_balloon_color_stats
df = conn.execute(f"""
    SELECT player, balloon_color, balloon_pops, points_by_color, bonus_hits
    FROM {CATALOG_NAME}.SILVER.DT_BALLOON_COLOR_STATS
    ORDER BY player, points_by_color DESC NULLS LAST
    LIMIT 10
""").df()
print("dt_balloon_color_stats:")
print(df.to_string(index=False))

In [ ]:
# dt_realtime_scores
df = conn.execute(f"""
    SELECT player, total_score, window_start, window_end
    FROM {CATALOG_NAME}.SILVER.DT_REALTIME_SCORES
    ORDER BY window_start DESC, player
    LIMIT 10
""").df()
print("dt_realtime_scores:")
print(df.to_string(index=False))

In [ ]:
# dt_balloon_colored_pops
df = conn.execute(f"""
    SELECT player, balloon_color, balloon_pops, window_start, window_end
    FROM {CATALOG_NAME}.SILVER.DT_BALLOON_COLORED_POPS
    ORDER BY window_start DESC, player, balloon_color
    LIMIT 10
""").df()
print("dt_balloon_colored_pops:")
print(df.to_string(index=False))

In [ ]:
# dt_color_performance_trends
df = conn.execute(f"""
    SELECT balloon_color, avg_score_per_pop, total_pops, window_start, window_end
    FROM {CATALOG_NAME}.SILVER.DT_COLOR_PERFORMANCE_TRENDS
    ORDER BY window_start DESC, avg_score_per_pop DESC NULLS LAST
    LIMIT 10
""").df()
print("dt_color_performance_trends:")
print(df.to_string(index=False))

## Summary

You just queried Snowflake-managed Dynamic Iceberg Tables directly from DuckDB using
the Horizon Iceberg REST Catalog (HIRC). Key takeaways:

1. **No data copy** — DuckDB reads Iceberg files from S3 using vended credentials from Snowflake
2. **PAT authentication** — OAuth2 client credentials flow with a scoped PAT
3. **Uppercase catalog names** — HIRC/Polaris is case-sensitive; use `DATABASE.upper()`
4. **Dynamic Table grants** — `ON ALL TABLES` skips DTs; use `ON ALL DYNAMIC TABLES`
5. **Multi-engine access** — the same Iceberg data is queryable from Snowflake, DuckDB, Spark, etc.